# Generating Synthetic Data (Open Source / Ollama edition)

This is the Week 3 exercise from the slide:

1. Write a model that can generate datasets
2. Use a variety of models and prompts for diverse outputs
3. Create a Gradio UI for your product

Everything below runs **entirely on your own machine** via [Ollama](https://ollama.com) - no OpenAI or Anthropic
API keys, no per-token cost, and no business data ever leaves your laptop. That last point is the actual reason
this matters commercially: if you can generate a *stand-in* dataset locally, you can prototype and test against
it before anyone has to trust a third party with the real thing.

**A note on "used commercially":** every model below ships under a licence that permits commercial use
(check the table further down), but that covers the *model weights*, not automatically everything they produce.
Before shipping generated data:
- Keep the safety instructions in the prompts below (no real names, no real company data, no copyrighted text) -
  they are there to stop a model from memorising and regurgitating something it saw in training.
- Skim a sample of the output yourself. Small local models occasionally produce a name or detail that reads as
  suspiciously specific; regenerate that row if so.
- This notebook produces *structurally* realistic data (right shapes, right ranges, plausible-looking values).
  It is not a statistically rigorous simulation of your real data's distributions - fine for UI demos, load
  testing and prototyping; not a substitute for real data when statistical fidelity matters (e.g. training a
  model you'll ship).

**Prerequisites:** Ollama running locally, with at least one of these pulled:

```
ollama pull phi3
ollama pull gpt-oss
ollama pull gemma3:270m
```


In [ ]:
# imports

import os
import re
import json
import tempfile
from itertools import count

import requests
import pandas as pd
from openai import OpenAI
import gradio as gr

## Initialization

Change `OLLAMA_PORT` if your Ollama listens somewhere other than the default 11434.

`MODEL_INFO` is the set of models this notebook offers in the UI, each with its licence noted so you can judge
commercial fit for yourself. All three are pulled locally already if you followed the prerequisites above.

**On a CPU-only machine, `gpt-oss:latest` is genuinely slow** - on this machine, a one-word reply took ~45
seconds, and a full batch of structured records can take several minutes. It's included because the quality
is noticeably better, but it's left unchecked by default in the UI below; the inline demos in this notebook
use only `phi3` and `gemma3:270m` so the notebook itself runs in a reasonable time top-to-bottom. Turn on
`gpt-oss:latest` in the UI when you want quality and can afford to wait, or any time you have a CUDA GPU.

In [ ]:
OLLAMA_PORT = 11434
OLLAMA_HOST = f"http://localhost:{OLLAMA_PORT}"

# model tag -> (human label, licence). All three permit commercial use, but always re-check
# the current licence text for the exact version you deploy with.
MODEL_INFO = {
    "phi3": ("Phi-3 Mini (3.8B) - fast, reliable at structured JSON", "MIT"),
    "gemma3:270m": ("Gemma 3 270M - very fast, unreliable past ~3 columns", "Gemma"),
    "gpt-oss:latest": ("gpt-oss (20B) - best quality, but slow with no GPU", "Apache-2.0"),
}

openai = OpenAI(base_url=f"{OLLAMA_HOST}/v1", api_key="ollama")

In [ ]:
# Sanity check: is Ollama up, and are the models we want to offer actually pulled?

try:
    installed = {m["name"] for m in requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5).json()["models"]}
except Exception as e:
    raise SystemExit(f"Could not reach Ollama at {OLLAMA_HOST} - is it running? ({e})")

print(f"Ollama is up at {OLLAMA_HOST}.")
for tag, (label, licence) in MODEL_INFO.items():
    found = tag in installed or any(name.startswith(f"{tag.split(':')[0]}:") for name in installed)
    print(f"  {'OK ' if found else 'MISSING'}  {tag:<16} {licence:<10} {label}")
    if not found:
        print(f"           -> run:  ollama pull {tag}")

## Step 1: Write a model that can generate a dataset

A single call to a 3-8B local model can generate a handful of rows reliably, but asking for a large,
consistently-shaped dataset in one shot is exactly where small models fall apart - they lose track of the
schema, invent new columns, or just trail off. So we split the job in two:

1. **`infer_schema`** - one call that turns a plain-English business description into a column list.
2. **`generate_batch`** - many small calls (one per batch) that each produce a handful of rows against that
   fixed schema, which we'll later spread across different models and prompts.

Both ask the model for a single JSON object (`response_format={"type": "json_object"}` - Ollama enforces this
for models that support it) rather than free text, which is far more reliable to parse than asking for raw CSV
or a bare JSON array.

In [ ]:
SAFETY_RULES = (
    "Every record must be entirely fictional. Never reuse a real, identifiable person's name, "
    "a real company's proprietary data, or copyrighted text. Invent plausible-sounding but fake "
    "names, emails and identifiers instead."
)


def _extract_json_object(text):
    """Strip markdown fences if present, then parse the first {...} block as JSON."""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        text = match.group(0)
    return json.loads(text)


def infer_schema(business, model, hint_columns=""):
    """Ask one model to propose a column list for this business. Retries once at a lower
    temperature, then falls back to a generic schema - small models occasionally emit a
    stray or misnamed key, and schema quality matters more here than in any single batch."""
    system = "You are a data architect. You output only a single valid JSON object, no prose, no markdown fences."
    user = (
        f"A business is described as: {business}\n"
        "Propose a compact tabular schema for a synthetic dataset about this business. "
        'Return {"columns": [{"name": ..., "type": ...}, ...]} with 4 to 8 columns.'
    )
    if hint_columns:
        user += f"\nThe user specifically asked for these columns to be included: {hint_columns}."
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]

    for temperature in (0.3, 0.1):
        response = openai.chat.completions.create(
            model=model, messages=messages,
            response_format={"type": "json_object"}, temperature=temperature,
        )
        try:
            data = _extract_json_object(response.choices[0].message.content)
            columns = [c.get("name") for c in data.get("columns", []) if isinstance(c, dict) and c.get("name")]
            columns = list(dict.fromkeys(str(c) for c in columns))  # de-dupe, keep order
            if len(columns) >= 3:
                return columns
        except (json.JSONDecodeError, AttributeError):
            continue
    return ["id", "name", "category", "value", "date", "notes"]

In [ ]:
def generate_batch(business, schema, model, batch_size, persona_hint):
    """Ask one model for `batch_size` fictional records matching `schema`."""
    system = (
        "You are a synthetic data generator for commercial product testing. You output only a "
        f"single valid JSON object, no prose, no markdown fences. {SAFETY_RULES}"
    )
    user = (
        f"Business: {business}\n"
        f"Columns (use exactly these keys, in this order): {schema}\n"
        f"Generate exactly {batch_size} realistic but fictional records as a JSON object: "
        '{"records": [{...}, ...]}. Every record is a flat object - no nested objects or extra '
        'top-level keys besides "records".\n'
        f"Style guidance for this batch: {persona_hint}"
    )
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]

    # Small local models occasionally emit malformed JSON. One low-temperature retry recovers
    # almost all of those cases without the caller ever needing to know it happened.
    for temperature in (0.8, 0.2):
        response = openai.chat.completions.create(
            model=model, messages=messages,
            response_format={"type": "json_object"}, temperature=temperature,
        )
        try:
            data = _extract_json_object(response.choices[0].message.content)
            records = data.get("records", [])
            if isinstance(records, list) and records:
                return [r for r in records if isinstance(r, dict)]
        except (json.JSONDecodeError, AttributeError):
            continue
    return []

Quick test - infer a schema, then generate one batch against it:

In [ ]:
business = "A boutique online store selling handmade leather wallets"
schema = infer_schema(business, "phi3")
print("Schema:", schema)

rows = generate_batch(business, schema, "phi3", 5, "budget-conscious young professionals in a big city")
pd.DataFrame(rows).reindex(columns=schema)

## Step 2: Use a variety of models and prompts for diverse outputs

One model, asked the same way every time, tends to converge on similar names, numbers and phrasing across an
entire dataset - real-world data isn't that uniform. So `generate_dataset` spreads the work across:

- **multiple models** (round-robin through whichever ones you selected), and
- **multiple persona/style hints** per batch (different customer types, regions, tones),

so batch 3 might be Phi-3 writing for budget shoppers while batch 4 is gpt-oss writing for wealthy retirees.
Exact-duplicate rows (which small models do occasionally repeat) are dropped, and if a batch comes back short
the shortfall just rolls into the next one - so a flaky model or a bad prompt degrades the mix rather than
failing the whole run.

In [ ]:
PERSONA_HINTS = [
    "budget-conscious young professionals in a big city",
    "wealthy retirees shopping for gifts",
    "rural small-business owners restocking supplies",
    "students shopping on a tight budget",
    "international customers ordering for shipment abroad",
    "repeat customers making a routine reorder",
]


def generate_dataset(business, models, num_records, hint_columns="", batch_size=8, log_fn=print):
    schema = infer_schema(business, models[0], hint_columns)
    log_fn(f"Schema: {schema}")

    rows = []
    batch_index = 0
    max_attempts = (num_records // batch_size + 2) * 2  # generous ceiling so a flaky run can't spin forever
    while len(rows) < num_records and batch_index < max_attempts:
        model = models[batch_index % len(models)]
        persona = PERSONA_HINTS[batch_index % len(PERSONA_HINTS)]
        size = min(batch_size, num_records - len(rows))

        batch_rows = generate_batch(business, schema, model, size, persona)
        log_fn(f"batch {batch_index}: model={model} persona='{persona}' -> {len(batch_rows)} rows")
        rows.extend(batch_rows)
        batch_index += 1

    # de-dupe exact-duplicate records while preserving generation order
    seen, deduped = set(), []
    for r in rows:
        key = json.dumps(r, sort_keys=True)
        if key not in seen:
            seen.add(key)
            deduped.append(r)
    if len(deduped) < len(rows):
        log_fn(f"Dropped {len(rows) - len(deduped)} exact-duplicate rows")

    df = pd.DataFrame(deduped[:num_records]).reindex(columns=schema)
    return df, schema

Try it with two different models mixed together - watch the log to see which model produced which batch:

In [ ]:
df, schema = generate_dataset(
    business="A boutique online store selling handmade leather wallets",
    models=["phi3", "gemma3:270m"],
    num_records=15,
)
df

## Step 3: Package the result for download

A Gradio `gr.File` output needs an actual file on disk, so we write the finished DataFrame out as CSV or JSON
to a temp file and hand back the path.

In [ ]:
_file_counter = count()


def save_dataset(df, fmt):
    path = os.path.join(tempfile.gettempdir(), f"synthetic_dataset_{next(_file_counter)}.{fmt.lower()}")
    if fmt == "CSV":
        df.to_csv(path, index=False)
    else:
        df.to_json(path, orient="records", indent=2)
    return path

## Step 4: Gradio UI

A `gr.Blocks` layout: describe the business, optionally steer the schema, pick which models to mix in, choose
how many rows and which export format, then generate. The log panel makes the model/prompt diversity from
Step 2 visible rather than hidden inside a black box.

In [ ]:
def run_generation(business, hint_columns, models_selected, num_records, fmt):
    if not business.strip():
        return None, None, "Please describe the business first."
    if not models_selected:
        return None, None, "Please select at least one model."

    log_lines = []
    df, _schema = generate_dataset(
        business=business,
        models=models_selected,
        num_records=int(num_records),
        hint_columns=hint_columns,
        log_fn=log_lines.append,
    )
    path = save_dataset(df, fmt)
    log_lines.append(f"Saved {len(df)} rows to {path}")
    return df, path, "\n".join(log_lines)


with gr.Blocks(title="Synthetic Dataset Generator") as ui:
    gr.Markdown("# Synthetic Dataset Generator\n_100% local, open-weight models via Ollama_")

    with gr.Row():
        with gr.Column():
            business = gr.Textbox(
                label="Describe the business",
                placeholder="e.g. A boutique online store selling handmade leather wallets",
                lines=3,
            )
            hint_columns = gr.Textbox(
                label="Columns to include (optional - leave blank to let the model decide)",
                placeholder="e.g. order_id, customer_name, item, price, order_date",
            )
            models_selected = gr.CheckboxGroup(
                choices=[(f"{label} [{licence}]", tag) for tag, (label, licence) in MODEL_INFO.items()],
                value=["phi3", "gemma3:270m"],
                label="Models to mix in (round-robin across batches for varied output) - "
                      "gpt-oss is unchecked by default because it's slow without a GPU",
            )
            num_records = gr.Slider(10, 200, value=30, step=10, label="Number of records")
            fmt = gr.Radio(["CSV", "JSON"], value="CSV", label="Export format")
            generate_button = gr.Button("Generate dataset", variant="primary")

        with gr.Column():
            output_table = gr.Dataframe(label="Preview", wrap=True)
            output_file = gr.File(label="Download full dataset")
            output_log = gr.Textbox(label="Generation log", lines=8)

    generate_button.click(
        fn=run_generation,
        inputs=[business, hint_columns, models_selected, num_records, fmt],
        outputs=[output_table, output_file, output_log],
    )

ui.launch(inbrowser=True)

# Exercises and Business Applications

- **Bigger and better models**: swap in `mistral` or `qwen2.5` (`ollama pull mistral`) - both Apache-2.0 and
  generally sharper at following the schema than Phi-3, at the cost of a slower run on CPU.
- **Schema validation**: right now a batch that drifts off-schema just gets `reindex`-ed with blanks. Add a
  check that re-queues a batch whose keys don't match `schema` instead of silently accepting it.
- **Statistical realism**: this notebook optimises for *variety*, not *distribution fidelity*. For anything
  downstream of "looks plausible in a demo", pair this with a library like `faker` for the fields that need to
  be genuinely well-distributed (names, addresses, dates), and reserve the LLM for the fields that need
  business-specific judgement.
- **Bigger runs**: `num_records` above tops out at 200 for interactive use. For a large offline batch job, call
  `generate_dataset` directly from a script instead of through Gradio, and consider caching/re-using an
  inferred `schema` across runs for the same business so every export stays consistent.

The commercial angle is the point, not an afterthought: because this whole pipeline runs locally, you can
generate stand-in data for a business domain you'd never send to a third-party API, and every model here ships
under a licence permissive enough to use in a real product - always double-check the licence text for whichever
exact model and version you end up shipping with.